In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

print("Loading dataset...")

data = pd.read_csv("fraudTest.csv")

print("Dataset loaded successfully.")
print("Shape:", data.shape)
print("\nFirst 5 rows:")
print(data.head())

print("\nClass Distribution:")
print(data['is_fraud'].value_counts())
print("(0 = Legitimate, 1 = Fraud)")

Loading dataset...
Dataset loaded successfully.
Shape: (201789, 23)

First 5 rows:
   Unnamed: 0 trans_date_trans_time            cc_num  \
0           0   2020-06-21 12:14:25  2291163933867244   
1           1   2020-06-21 12:14:33  3573030041201292   
2           2   2020-06-21 12:14:53  3598215285024754   
3           3   2020-06-21 12:15:15  3591919803438423   
4           4   2020-06-21 12:15:17  3526826139003047   

                               merchant        category    amt   first  \
0                 fraud_Kirlin and Sons   personal_care   2.86    Jeff   
1                  fraud_Sporer-Keebler   personal_care  29.84  Joanne   
2  fraud_Swaniawski, Nitzsche and Welch  health_fitness  41.28  Ashley   
3                     fraud_Haley Group        misc_pos  60.05   Brian   
4                 fraud_Johnston-Casper          travel   3.19  Nathan   

       last gender                       street  ...      lat      long  \
0   Elliott      M            351 Darlene Green  ...  

In [2]:
print("Preprocessing...")

data = data[['amt', 'zip', 'lat', 'long', 'city_pop', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud']]
data = data.dropna()

scaler = StandardScaler()
data['amt'] = scaler.fit_transform(data[['amt']])

X = data.drop(columns=['is_fraud'])
y = data['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Data split completed.")
print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Preprocessing...
Data split completed.
Training samples: 161430
Testing samples: 40358


In [3]:
print("Training and Comparing Models...")

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

for name, clf in models.items():
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print(f"\n{name} Accuracy: {round(acc * 100, 2)} %")
    print(classification_report(y_test, preds, target_names=["Legitimate", "Fraud"]))

print("Best model is Random Forest for fraud detection.")
model = models["Random Forest"]

Training and Comparing Models...

Logistic Regression Accuracy: 99.57 %
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     40185
       Fraud       0.00      0.00      0.00       173

    accuracy                           1.00     40358
   macro avg       0.50      0.50      0.50     40358
weighted avg       0.99      1.00      0.99     40358



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



Decision Tree Accuracy: 99.55 %
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     40185
       Fraud       0.48      0.56      0.52       173

    accuracy                           1.00     40358
   macro avg       0.74      0.78      0.76     40358
weighted avg       1.00      1.00      1.00     40358


Random Forest Accuracy: 99.8 %
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     40185
       Fraud       0.89      0.60      0.72       173

    accuracy                           1.00     40358
   macro avg       0.94      0.80      0.86     40358
weighted avg       1.00      1.00      1.00     40358

Best model is Random Forest for fraud detection.


In [4]:
print("\nFraud Detection")

print("\nEnter transaction details:")
amt = float(input("Transaction Amount: "))
zip_code = float(input("ZIP Code: "))
lat = float(input("Latitude: "))
long_ = float(input("Longitude: "))
city_pop = float(input("City Population: "))
unix_time = float(input("Unix Time: "))
merch_lat = float(input("Merchant Latitude: "))
merch_long = float(input("Merchant Longitude: "))

import numpy as np

amt_scaled = scaler.transform([[amt]])[0][0]

user_input = np.array([[amt_scaled, zip_code, lat, long_, city_pop, unix_time, merch_lat, merch_long]])
result = model.predict(user_input)

if result[0] == 1:
    print("\nTransaction is FRAUDULENT 🚨")
else:
    print("\nTransaction is LEGITIMATE ✅")


Fraud Detection

Enter transaction details:
Transaction Amount: 29.84
ZIP Code: 29209
Latitude: 33.9659
Longitude: -80.9355
City Population: 333497
Unix Time: 1371816865
Merchant Latitude: 33.986391
Merchant Longitude: -81.200714

Transaction is LEGITIMATE ✅


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
